# OSRT v6 — interactive inference with token stats

Type any prompt, get the generation plus **TTFT, prefill throughput, decode tok/s, end-to-end tok/s, peak VRAM**.

- **GPU**: any Colab CUDA GPU (A100 / L4 / RTX 6000 Pro). bf16 + `torch.compile` throughout.
- **Secrets**: add `HF_TOKEN` in Colab's key sidebar (🔑) — needed for the private checkpoint repo `HallD/osrt-v6-ckpt`.
- **First generation is slow** (multi-minute): `torch.compile` traces the model. `dynamic=True` means it compiles ONCE, not per prompt length. Every generation after that runs at full speed.
- Measured reference points (batch 1, 20 Sinkhorn iters, full 6 loops): **A100 ~92 tok/s · H100 ~119 · B200 ~136**.

> RTX 6000 Pro note (sm_120 consumer Blackwell): the MoE prefill path uses `torch._grouped_mm`, proven on sm_90/sm_100 but not yet on sm_120. Cell 2 runs a tiny smoke forward first — if it errors there, report back before burning time on workarounds.

In [ ]:
# ── 1. Setup: repo code + deps + HF auth ─────────────────────────────
import os, sys

BRANCH = "feat/sft-harvest"  # has the decode-perf work (static cache, CUDA graphs)
if not os.path.isdir("/content/osrt"):
    !git clone -q --depth 1 -b {BRANCH} https://github.com/CodeHalwell/OSRT-605M-A269M.git /content/osrt
%pip -q install -U huggingface_hub transformers
sys.path.insert(0, "/content/osrt/src")

from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

import torch
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> GPU"
print("GPU:", torch.cuda.get_device_name(0),
      f"| sm_{torch.cuda.get_device_capability(0)[0]}{torch.cuda.get_device_capability(0)[1]}",
      "| torch", torch.__version__)

In [ ]:
# ── 2. Load checkpoint + tokenizer, smoke forward ────────────────────
CKPT = "osrt_v5_midtrain3_final.pt"  # or any osrt_v5_midtrain3_step_*.pt / SFT ckpt name
HF_REPO = "HallD/osrt-v6-ckpt"

from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer
from osrt.model import OSRTForCausalLM
from osrt.presets import build_config

path = hf_hub_download(HF_REPO, CKPT, repo_type="model")
# v6_tokenizer_export = the 65K v6 contract (matches the ckpt embedding).
# The repo's tokenizer/ dir is a STALE 32K artifact — do not use it.
tok = AutoTokenizer.from_pretrained("/content/osrt/v6_tokenizer_export")
assert len(tok) == 65536, f"wrong tokenizer: {len(tok)} tokens, expected 65536"
cfg = build_config(
    vocab_size=len(tok), real_vocab_size=len(tok),
    bos_token_id=tok.bos_token_id, eos_token_id=tok.eos_token_id,
    pad_token_id=tok.pad_token_id, fused_cross_entropy_chunks=8,
)
device = torch.device("cuda")
model = OSRTForCausalLM(cfg).to(device)
sd = torch.load(path, map_location=device, weights_only=True)
sd = sd.get("model_state_dict", sd)
missing, unexpected = model.load_state_dict(sd, strict=False)
assert not missing and not unexpected, f"state mismatch: {missing[:3]} {unexpected[:3]}"
model.eval()

# smoke forward BEFORE compiling — surfaces any arch-support problem
# (e.g. torch._grouped_mm on new GPUs) with a readable eager traceback.
with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.bfloat16):
    ids = torch.tensor([[tok.bos_token_id] + tok.encode("Hello", add_special_tokens=False)], device=device)
    model(ids)
print(f"loaded {CKPT} — smoke forward OK")

In [ ]:
# ── 3. Optimize + warmup (the slow cell — compile happens here) ──────
COMPILE = True          # fullgraph/dynamic torch.compile (~2.5x)
CUDA_GRAPHS = True      # reduce-overhead decode callable (further ~1.5-4x at b1)
CACHE_IMPL = "static"   # "static" (speed mode, needs CUDA_GRAPHS for full effect) | "latent"

model.optimize_for_inference(compile_model=COMPILE, reduce_overhead=CUDA_GRAPHS)

import time
print("warmup generation (compiles on first call — this can take several minutes)...")
t0 = time.perf_counter()
with torch.amp.autocast("cuda", dtype=torch.bfloat16):
    model.generate(ids, max_new_tokens=8, cache_impl=CACHE_IMPL)
    model.generate(ids, max_new_tokens=8, cache_impl=CACHE_IMPL)  # 2nd = graph capture/replay warm
torch.cuda.synchronize()
print(f"warm ({time.perf_counter() - t0:.0f}s). Everything from here runs at full speed.")

In [ ]:
# ── 4. Timed generation helper ───────────────────────────────────────
import time

# Base (pre-SFT) checkpoints are raw continuation models -> leave False.
# For SFT checkpoints set True to wrap prompts in the v6 chat contract.
CHAT_TEMPLATE = False
SYSTEM = "You are a helpful assistant. Think step by step in <|think|>...<|/think|>, then answer in <|answer|>...<|/answer|>."

def timed_generate(prompt: str, max_new_tokens: int = 256, temperature: float = 0.8,
                   top_p: float = 0.95, top_k: int = 40, repetition_penalty: float = 1.2,
                   num_loops: int | None = None) -> dict:
    """One generation, instrumented. TTFT via a 1-token call (prefill + first
    decode step); steady-state decode rate from the remainder of the full call."""
    text = (f"<|system|>{SYSTEM}<|user|>{prompt}<|assistant|>" if CHAT_TEMPLATE else prompt)
    ids = [tok.bos_token_id] + tok.encode(text, add_special_tokens=False)
    inp = torch.tensor([ids], device=device)
    kw = dict(temperature=temperature, top_p=top_p, top_k=top_k,
              repetition_penalty=repetition_penalty, eos_token_id=tok.eos_token_id,
              cache_impl=CACHE_IMPL, num_loops=num_loops)
    torch.cuda.reset_peak_memory_stats()

    torch.cuda.synchronize(); t0 = time.perf_counter()
    with torch.amp.autocast("cuda", dtype=torch.bfloat16):
        model.generate(inp, max_new_tokens=1, **kw)
    torch.cuda.synchronize(); ttft = time.perf_counter() - t0

    torch.cuda.synchronize(); t0 = time.perf_counter()
    with torch.amp.autocast("cuda", dtype=torch.bfloat16):
        out = model.generate(inp, max_new_tokens=max_new_tokens, **kw)
    torch.cuda.synchronize(); total = time.perf_counter() - t0

    gen = out[0, len(ids):].tolist()
    n = len(gen)
    decode_tps = (n - 1) / max(1e-9, total - ttft) if n > 1 else float("nan")
    return {
        "text": tok.decode(gen, skip_special_tokens=False),
        "prompt_tokens": len(ids), "new_tokens": n,
        "ttft_ms": ttft * 1e3,
        "prefill_tps": len(ids) / ttft,          # ~ (TTFT includes 1 decode step)
        "decode_tps": decode_tps,                 # steady-state
        "e2e_tps": n / total, "total_s": total,
        "peak_vram_gb": torch.cuda.max_memory_allocated() / 2**30,
    }

def show(r: dict) -> None:
    print(r["text"])
    print("─" * 72)
    print(f"prompt {r['prompt_tokens']} tok | generated {r['new_tokens']} tok | "
          f"TTFT {r['ttft_ms']:.0f} ms | prefill ~{r['prefill_tps']:.0f} tok/s | "
          f"decode {r['decode_tps']:.1f} tok/s | e2e {r['e2e_tps']:.1f} tok/s | "
          f"{r['total_s']:.2f}s total | peak VRAM {r['peak_vram_gb']:.1f} GB")

show(timed_generate("The three laws of thermodynamics state that", max_new_tokens=128))

In [ ]:
# ── 5. Interactive loop — type prompts, empty line or 'quit' to stop ─
MAX_NEW_TOKENS = 256
TEMPERATURE = 0.8   # 0.0 = greedy

while True:
    try:
        prompt = input("\nprompt> ").strip()
    except (EOFError, KeyboardInterrupt):
        break
    if not prompt or prompt.lower() in {"quit", "exit"}:
        break
    show(timed_generate(prompt, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE))

## Notes

- **TTFT** = time to first token (prefill + one decode step + Python overhead). **decode tok/s** is the steady-state rate — the number to compare against the A100/H100/B200 references.
- **Greedy vs sampled**: `TEMPERATURE = 0.0` is deterministic; the compiled path is ppl-identical to eager but not bit-reproducible on near-tie logits.
- **This is the *base* model** (midtrain3, pre-SFT): expect continuation behaviour, not chat. When testing an SFT checkpoint, set `CKPT` in cell 2 and `CHAT_TEMPLATE = True` in cell 4.
- `num_loops=` in `timed_generate` (1–6) trades quality for speed (~1.5x per 2 loops dropped) — quality is NOT gated below 6; for probing only.
- If the runtime disconnects, rerun cells 1–3 (compile cache is ephemeral on Colab, so the warmup cost repeats).